# U02 · Fundamentos y datos clínicos: EDA y limpieza

> **Para profesionales sanitarios.** No hace falta saber programar. El código ya está escrito y **comentado**; tu trabajo es **entender qué hace y por qué**, con criterio clínico. Ejecuta las celdas de arriba abajo (botón **▶** o **Mayús + Enter**).

> ⚠️ **Todos los datos de este cuaderno son SINTÉTICOS.** Se generan con código reproducible y su única función es didáctica. **No representan pacientes reales** y **nunca** deben usarse en clínica.

## Qué vamos a hacer

Los datos clínicos **rara vez llegan limpios**. En este cuaderno trabajamos con un histórico de pacientes "sucio" (`pacientes_sucio.csv`) —el tipo de volcado imperfecto que sale de una historia clínica real— y aprendemos a **mirar, diagnosticar y corregir** sus problemas, uno a uno:

1. **Asomarnos** a los datos (forma, primeras filas, tipos) y detectar qué está mal.
2. **Categorías inconsistentes** de `sexo` y `tabaquismo` → normalizar.
3. **Glucemia con unidades mezcladas** (mg/dL y mmol/L) → reconvertir.
4. **Texto colado** en columnas numéricas (`"desconocido"`, `"N/D"`) → pasar a número.
5. **Nulos no aleatorios** en `hdl`/`colesterol` → imputar con criterio.
6. **Valores imposibles** (edad 250, tensión negativa, IMC gigante) → filtrar.
7. **Duplicados** → eliminar.

Y cerramos con una idea que vertebra todo el curso: **un histórico sucio genera modelos sesgados**, y eso, en salud, se traduce en **desigualdad ante el paciente**.

> 💡 La secuencia mental es siempre la misma: **mirar → diagnosticar → corregir con reglas de dominio clínico → verificar el antes/después.**


## 1. Generamos los datos (esta celda se ejecuta sola)

La primera celda de todos los cuadernos del curso **genera los ficheros sintéticos** por sí sola: no hay que descargar nada. Entre ellos crea `pacientes_sucio.csv`, la versión con problemas de calidad que vamos a limpiar. Ejecútala y espera el mensaje de confirmación.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 2. Primer vistazo: ¿qué pinta tienen los datos?

Antes de tocar nada, **nos asomamos**. Lo primero con cualquier tabla nueva es responder a tres preguntas: **¿cuántas filas y columnas tiene? (`shape`)**, **¿qué aspecto tienen las primeras filas? (`head`)** y **¿de qué tipo es cada columna? (`dtypes`)**. Empecemos por el tamaño y las primeras filas.


In [ ]:
import pandas as pd

df_sucio = pd.read_csv("pacientes_sucio.csv")
print("Filas y columnas:", df_sucio.shape)
df_sucio.head()

**Lo que vemos:** una tabla de unas 20.000 filas (un paciente por fila) y 15 columnas. A simple vista, en `sexo` conviven `M`, `Masculino`, `Mujer`… y en `glucemia` aparecen valores con **coma** (`5,5`). Ya intuimos problemas. Vamos a confirmarlos mirando el **tipo** de cada columna.


### Los tipos de columna: la primera señal de alarma

`dtypes` nos dice cómo ha interpretado pandas cada columna. Una columna que **debería ser numérica** pero aparece como `object` (texto) es una bandera roja: significa que hay algo no numérico dentro (una coma, una palabra…).


In [ ]:
df_sucio.dtypes

**Lo que vemos —y ya es un diagnóstico:** `glucemia`, `colesterol_total` y `ta_diastolica` figuran como **`object` (texto)**, cuando deberían ser **numéricas**. Eso solo puede significar una cosa: dentro hay caracteres que no son números (comas decimales, o palabras como `"desconocido"`). Los demás campos numéricos (`edad`, `imc`, `ta_sistolica`…) sí son numéricos, pero eso no garantiza que sus valores sean *sensatos*: lo comprobaremos.


### El resumen estadístico: `describe`

`describe(include="all")` da, de un vistazo, el mínimo, el máximo, la media y los valores más frecuentes de cada columna. Es un detector de rarezas: buscamos **mínimos o máximos imposibles** (una edad de 250, una tensión negativa) y **categorías que deberían ser pocas pero aparecen muchas**.


In [ ]:
df_sucio.describe(include="all").T

**Lo que vemos:** dos hallazgos que saltan a la vista. En `edad`, el **máximo** es absurdamente alto (más de 120 años). En las columnas de texto (`sexo`, `tabaquismo`) el número de **valores únicos** es mucho mayor de lo que debería: `sexo` debería tener **2** categorías y `tabaquismo` **3**, pero hay muchas más. Esto nos marca la lista de tareas. Vamos una por una.


## 3. Categorías inconsistentes (I): `sexo`

Empecemos por lo que parece más fácil. La variable `sexo` debería tener **dos** valores (`M`/`F`), pero veamos cuántos tiene en realidad. `value_counts()` cuenta cuántas veces aparece cada valor.


In [ ]:
df_sucio["sexo"].value_counts(dropna=False)

**Lo que vemos:** ¡media docena de formas de decir lo mismo! `Masculino`, `Hombre`, `H`, `M`, `m` (varón) y `Mujer`, `Femenino`, `F`, `f` (mujer). Todas significan solo dos cosas. Hay que **unificarlas** a `M`/`F`.

> ⚠️ **Normalizar no es interpretar.** Poner todo en minúsculas y sin espacios (`strip().lower()`) unifica el *formato*, pero no decide qué *significa* cada valor. En español, una `M` suelta podría ser ambigua (¿*Masculino* o *Mujer*?). La norma de oro: **primero normaliza el formato, después mapea el significado con una regla explícita.** En este dataset sabemos, por cómo se generó, que `M`/`m` provienen de "masculino" y `F`/`f` de "femenino", así que podemos mapear con seguridad —pero dejando constancia de la decisión.


Aplicamos la regla: normalizamos el texto (minúsculas, sin espacios) y **mapeamos** cada variante a `M` o `F`.


In [ ]:
# Regla de dominio: unificamos el formato y luego mapeamos el significado
mapa_sexo = {
    "m": "M", "masculino": "M", "hombre": "M", "h": "M",   # varón
    "f": "F", "femenino": "F", "mujer": "F",                # mujer
}
df = df_sucio.copy()                                        # trabajamos sobre una copia
sexo_norm = df["sexo"].astype(str).str.strip().str.lower()  # normalizar formato
df["sexo"] = sexo_norm.map(mapa_sexo)                       # mapear significado

print("ANTES:", sorted(df_sucio["sexo"].astype(str).unique()))
print("DESPUÉS:", sorted(df["sexo"].dropna().unique()))
print("¿Quedan valores sin mapear (nulos)?:", int(df["sexo"].isna().sum()))

**Lo que vemos:** la columna pasa de media docena de variantes a exactamente **dos** (`M`, `F`), y **ningún** valor se quedó sin mapear. La variable ya es utilizable. Guardamos siempre el resultado en una **copia** (`df`) para no pisar el original (`df_sucio`): la trazabilidad es sagrada.


## 4. Categorías inconsistentes (II): `tabaquismo`

El mismo problema, y aquí importa aún más. `tabaquismo` debería tener tres categorías canónicas: **`nunca`**, **`ex`** (exfumador) y **`activo`**. Veamos qué llega.


In [ ]:
df_sucio["tabaquismo"].value_counts(dropna=False)

**Lo que vemos:** otra vez, variantes de las mismas tres ideas: `NUNCA`/`No fumador`, `ex`/`exfumador`/`Ex-fumador`, `activo`/`Fumador`/`SI`. Esto **no es cosmética**: en estos datos el hábito tabáquico marca un gradiente real de riesgo (los fumadores activos sufren más eventos que los exfumadores, y estos más que quienes nunca fumaron). Si la categoría llega sucia, ese gradiente clínico **se diluye** y el modelo pierde una señal valiosa. Normalizamos y mapeamos a las tres categorías canónicas.


In [ ]:
mapa_tabaco = {
    "nunca": "nunca", "no fumador": "nunca",
    "ex": "ex", "exfumador": "ex", "ex-fumador": "ex",
    "activo": "activo", "fumador": "activo", "si": "activo",
}
tab_norm = df["tabaquismo"].astype(str).str.strip().str.lower()
df["tabaquismo"] = tab_norm.map(mapa_tabaco)

print("DESPUÉS:", sorted(df["tabaquismo"].dropna().unique()))
print("¿Sin mapear (nulos)?:", int(df["tabaquismo"].isna().sum()))
df["tabaquismo"].value_counts()

**Lo que vemos:** la variable queda reducida a las **tres** categorías correctas, sin valores perdidos. Comprobemos que el **gradiente clínico** que mencionábamos existe de verdad en los datos: la proporción de eventos cardiovasculares (`evento_cv`) debería crecer de `nunca` a `activo`.


In [ ]:
# Proporción de eventos cardiovasculares por hábito tabáquico
tasa = df.groupby("tabaquismo")["evento_cv"].mean().reindex(["nunca", "ex", "activo"])
print((tasa * 100).round(1).astype(str) + " %")

**Lo que vemos:** la tasa de eventos **sube** de quienes nunca fumaron a los exfumadores y, más aún, a los fumadores activos. Ese orden coincide con el conocimiento clínico y confirma por qué merecía la pena limpiar la categoría: una señal real solo se aprovecha si está bien codificada.


## 5. Glucemia: unidades mezcladas (mmol/L → mg/dL)

Este es el problema más bonito, y el que mejor muestra por qué el **criterio clínico es insustituible**. Recordemos que `glucemia` figuraba como texto (`object`). Veamos algunos valores para entender por qué.


In [ ]:
# Miramos valores que contienen una coma (sospechosos)
con_coma = df_sucio["glucemia"].astype(str).str.contains(",", na=False)
df_sucio.loc[con_coma, "glucemia"].head(8).tolist()

**Lo que vemos:** valores como `"5,5"`, con **coma decimal**. Dos cosas ocurren aquí:

1. La **coma** impide que pandas lea la columna como número (por eso era `object`). Hay que sustituir la coma por punto y convertir a numérico.
2. Un valor de glucemia de **5,5 es imposible en mg/dL** (sería una hipoglucemia letal), pero **perfectamente normal en mmol/L**. La pista es puramente clínica: **conocemos el rango fisiológico.** Parte de los datos vino en **mmol/L** en lugar de mg/dL.

Primer paso: coma → punto y a numérico.


In [ ]:
# Sustituimos la coma decimal por punto y convertimos a número
glu = df["glucemia"].astype(str).str.replace(",", ".", regex=False)
df["glucemia"] = pd.to_numeric(glu, errors="coerce")   # lo no convertible pasaría a NaN
print("¿Es numérica ahora?:", df["glucemia"].dtype)
print("Mínimo:", round(df["glucemia"].min(), 1), " Máximo:", round(df["glucemia"].max(), 1))
print("Valores no convertibles (NaN):", int(df["glucemia"].isna().sum()))

**Lo que vemos:** ya es numérica, sin valores perdidos, pero con un rango sospechoso: el **mínimo** ronda 3 y el **máximo** 175. Un mismo campo con valores tan dispares es la firma de **dos unidades mezcladas**. La forma más clara de verlo es un **histograma**: si hay dos unidades, aparecerán **dos montañas** separadas.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df["glucemia"], bins=60, color="#00AEC7", edgecolor="white")
plt.axvline(30, color="#E4572E", linestyle="--", label="frontera ~30")
plt.xlabel("Glucemia (valor tal cual, sin corregir)")
plt.ylabel("Nº de pacientes")
plt.title("Glucemia antes de corregir: distribución bimodal (datos sintéticos)")
plt.legend()
plt.show()

**Lo que vemos:** exactamente **dos grupos**. Uno grande a la derecha (valores ~60–180: son **mg/dL**) y otro pequeño a la izquierda, pegado a cero (valores ~3–10: son **mmol/L**). Entre ambos hay un hueco claro alrededor de 30. Ese hueco nos da un **umbral seguro** para distinguirlos.

**¿Por qué 30?** Es una decisión clínica, no estadística. En mg/dL, una glucemia por debajo de ~40 sería una hipoglucemia mortal (el mínimo real de estos datos es 60 mg/dL). Y como referencia, el umbral de diabetes de 126 mg/dL equivale a **7,0 mmol/L**: los valores en mmol/L son necesariamente pequeños. Cualquier corte entre 20 y 60 funciona; usamos **30**.


Corregimos: los valores por debajo del umbral están en mmol/L y los **reconvertimos a mg/dL multiplicando por 18** (por ejemplo, 5,5 mmol/L × 18 = 99 mg/dL).


In [ ]:
# Regla de dominio: los valores en mmol/L (los bajos) se pasan a mg/dL (x18)
es_mmol = df["glucemia"] < 30
print("Valores detectados en mmol/L (a reconvertir):", int(es_mmol.sum()))

df.loc[es_mmol, "glucemia"] = df.loc[es_mmol, "glucemia"] * 18
print("Nuevo rango de glucemia (mg/dL):",
      round(df["glucemia"].min(), 1), "-", round(df["glucemia"].max(), 1))

In [ ]:
# Comprobamos con un histograma: ahora debe verse UNA sola distribución coherente
plt.figure(figsize=(8, 4))
plt.hist(df["glucemia"], bins=60, color="#00AEC7", edgecolor="white")
plt.axvline(126, color="#E4572E", linestyle="--", label="umbral diabetes (126 mg/dL)")
plt.xlabel("Glucemia (mg/dL)")
plt.ylabel("Nº de pacientes")
plt.title("Glucemia ya corregida a mg/dL (datos sintéticos)")
plt.legend()
plt.show()

**Lo que vemos:** ha desaparecido la montaña pegada a cero. Ahora hay **una única distribución** clínicamente coherente, centrada en torno a 100 mg/dL, con una cola hacia valores altos (los pacientes con hiperglucemia y diabetes, por encima de 126 mg/dL). Ninguna herramienta estadística "sabía" que había dos unidades: lo supo **quien conoce la fisiología**. Reconvertir bien es, literalmente, **traducir**.


## 6. Texto colado en columnas numéricas

Recordemos que `colesterol_total` y `ta_diastolica` también aparecían como texto. Veamos qué valores **no numéricos** contienen.


In [ ]:
for col in ["colesterol_total", "ta_diastolica"]:
    serie = df[col].astype(str)
    no_num = serie[pd.to_numeric(serie, errors="coerce").isna()]   # los que no son número
    print(col, "->", sorted(no_num.unique())[:6])

**Lo que vemos:** dentro de columnas que deberían ser puros números se han colado palabras como **`"desconocido"`** y **`"N/D"`**. Son, en realidad, **nulos disfrazados**: alguien anotó "no lo sé" en vez de dejar la casilla vacía, y eso obliga a pandas a leer toda la columna como texto. El tratamiento correcto es convertirlas a numérico **forzando que lo no válido se convierta en nulo** (`NaN`), y tratarlo después como un hueco más.


In [ ]:
for col in ["colesterol_total", "ta_diastolica"]:
    antes_texto = int(df[col].dtype == object)
    df[col] = pd.to_numeric(df[col], errors="coerce")   # 'desconocido'/'N/D' -> NaN
    print(f"{col}: ahora es {df[col].dtype}, con {int(df[col].isna().sum())} nulos")

**Lo que vemos:** ambas columnas ya son numéricas. Al convertir, los textos se han transformado en nulos, que se suman a los huecos que ya hubiera. Esto nos lleva de la mano al siguiente problema: **los valores ausentes**.


## 7. Valores ausentes (nulos): contar y, sobre todo, entender por qué faltan

No todos los huecos son iguales, y este es uno de los puntos donde un tratamiento ingenuo **inyecta sesgo**. Primero, ¿cuántos nulos hay en cada columna?


In [ ]:
nulos = df.isna().sum()
pct = (nulos / len(df) * 100).round(1)
resumen_nulos = pd.DataFrame({"nulos": nulos, "%": pct})
resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False)

**Lo que vemos:** las columnas con huecos son sobre todo **`hdl`** y **`colesterol_total`**. Ahora viene la pregunta clave, la que marca la diferencia entre limpiar bien o mal: **¿faltan al azar, o hay un patrón?** Un hueco que sigue un patrón (por ejemplo, que falte más en cierto grupo de pacientes) es **peligroso**, porque tratarlo mal desequilibra ese grupo. Comprobémoslo para `hdl` según la **edad**.


In [ ]:
# ¿Falta 'hdl' por igual en todas las edades? Comparamos jóvenes vs. el resto
# (nos ceñimos a edades plausibles para que las edades imposibles no distorsionen)
plausible = df["edad"].between(18, 120)
joven = plausible & (df["edad"] < 40)
resto = plausible & (df["edad"] >= 40)
print("Falta 'hdl' en <40 años :", round(df.loc[joven, "hdl"].isna().mean() * 100, 1), "%")
print("Falta 'hdl' en >=40 años:", round(df.loc[resto, "hdl"].isna().mean() * 100, 1), "%")

**Lo que vemos —y es importante:** falta **muchísimo más HDL en los pacientes jóvenes** (roughly el triple que en el resto). Esto **no es azar**: es un patrón (en la jerga, un nulo *no aleatorio*, MNAR). Refleja algo real de los sistemas sanitarios —a los jóvenes se les piden menos analíticas completas—. Y cambia por completo cómo debemos tratarlo. Veámoslo en una gráfica por tramos de edad para que quede grabado.


In [ ]:
# Tasa de HDL ausente por tramo de edad
df_pl = df[plausible].copy()
df_pl["tramo_edad"] = pd.cut(df_pl["edad"], bins=[18, 30, 40, 50, 65, 120],
                             labels=["18-30", "30-40", "40-50", "50-65", "65+"])
tasa_na = df_pl.groupby("tramo_edad")["hdl"].apply(lambda s: s.isna().mean() * 100)

plt.figure(figsize=(8, 4))
plt.bar(tasa_na.index.astype(str), tasa_na.values, color="#00AEC7", edgecolor="white")
plt.xlabel("Tramo de edad (años)")
plt.ylabel("% de pacientes sin dato de HDL")
plt.title("El HDL falta más en los jóvenes: un nulo NO aleatorio (datos sintéticos)")
plt.show()

**Lo que vemos:** la barra de los más jóvenes es claramente la más alta. Por eso las dos soluciones fáciles son **malas**:

- **Rellenar todos los huecos con la mediana global de HDL** empujaría a *todos* los jóvenes hacia el valor típico de la población general, **falseando justo al subgrupo peor medido**.
- **Eliminar las filas sin HDL** sería aún peor: expulsaríamos sistemáticamente a los pacientes jóvenes, y el modelo aprendería un mundo **sin ellos**.

La opción con criterio: **imputar informado por edad (y sexo)** —rellenar el hueco de cada paciente con la mediana de su propio grupo de edad y sexo, no con la de todo el mundo—. Y, sobre todo, **documentarlo**.


In [ ]:
# Imputación con criterio: mediana de HDL dentro del grupo (edad x sexo), no global
df["tramo_edad"] = pd.cut(df["edad"].clip(18, 120), bins=[18, 30, 40, 50, 65, 120],
                          labels=["18-30", "30-40", "40-50", "50-65", "65+"])
df["hdl"] = df.groupby(["tramo_edad", "sexo"])["hdl"].transform(lambda s: s.fillna(s.median()))
df["hdl"] = df["hdl"].fillna(df["hdl"].median())          # red de seguridad si algún grupo quedara vacío

# 'colesterol_total' sí falta de forma aproximadamente aleatoria: mediana global es razonable
df["colesterol_total"] = df["colesterol_total"].fillna(df["colesterol_total"].median())

print("Nulos restantes en hdl:", int(df["hdl"].isna().sum()))
print("Nulos restantes en colesterol_total:", int(df["colesterol_total"].isna().sum()))
df = df.drop(columns="tramo_edad")                        # variable auxiliar, ya no la necesitamos

**Lo que vemos:** cero nulos en ambas columnas. Pero fijémonos en la **decisión**: `hdl` se imputó **por grupo de edad y sexo** (porque su ausencia dependía de la edad), mientras que `colesterol_total`, cuyos huecos eran aproximadamente aleatorios, admite la mediana global. **El mismo problema (nulos) pide tratamientos distintos según *por qué* faltan.** Y quien use este dataset debe saber que, con los jóvenes, el HDL es un valor imputado y por tanto **menos fiable**.


## 8. Valores imposibles (outliers de dominio)

`describe` nos avisó de una edad máxima superior a 120. Rastreemos los valores **físicamente imposibles**: no son "extremos a discutir", son errores. Los buscamos con **rangos de dominio clínico**, no con un criterio estadístico ciego.


In [ ]:
# Contamos violaciones de rangos fisiológicos plausibles
imp_edad = (df["edad"] < 0) | (df["edad"] > 120)
imp_tas  = df["ta_sistolica"] <= 0
imp_imc  = (df["imc"] < 12) | (df["imc"] > 70)
print("Edades imposibles (>120 o <0):", int(imp_edad.sum()),
      "| ejemplos:", sorted(df.loc[imp_edad, "edad"].unique())[:5])
print("Tensión sistólica <= 0       :", int(imp_tas.sum()),
      "| ejemplos:", sorted(df.loc[imp_tas, "ta_sistolica"].unique())[:5])
print("IMC imposible (>70 o <12)    :", int(imp_imc.sum()),
      "| ejemplos:", sorted(df.loc[imp_imc, "imc"].round(1).unique())[:5])

**Lo que vemos:** un puñado de registros con edades de 150–260 años, tensiones **negativas** y valores de IMC de 80–180. Ninguno es un paciente real posible: son fallos de registro o de sensor. La regla clínica es clara: **edad entre 0 y 120, tensión positiva, IMC en un intervalo humano** (~12–70). Como son pocos y sin arreglo fiable, los **filtramos** (eliminamos esas filas).


In [ ]:
filas_antes = len(df)
df = df[~(imp_edad | imp_tas | imp_imc)].copy()
print("Filas antes :", filas_antes)
print("Filas después:", len(df), " (eliminadas", filas_antes - len(df), "con valores imposibles)")
print("Nuevos rangos ->  edad:", int(df["edad"].min()), "-", int(df["edad"].max()),
      "| ta_sistolica:", int(df["ta_sistolica"].min()), "-", int(df["ta_sistolica"].max()),
      "| imc:", round(df["imc"].min(), 1), "-", round(df["imc"].max(), 1))

**Lo que vemos:** los rangos vuelven a ser humanos: edades hasta ~95, tensiones positivas y en rango, IMC plausible. Hemos perdido muy pocas filas, y a cambio hemos quitado ruido que habría contaminado cualquier modelo. **Un IQR automático podría haber dejado pasar algunos de estos, o haber descartado valores altos pero reales**; el rango de dominio es más fiable.


## 9. Duplicados: filas y pacientes repetidos

Último problema. En un volcado real, los reprocesos generan **filas repetidas**. Además, aquí algunos `paciente_id` traen **espacios** al principio o al final (`" P00042 "`), lo que impide detectarlos como iguales. Primero limpiamos los identificadores; luego contamos duplicados **exactos**.


In [ ]:
df["paciente_id"] = df["paciente_id"].astype(str).str.strip()   # quitar espacios sobrantes
print("Filas duplicadas EXACTAS (todas las columnas iguales):", int(df.duplicated().sum()))

**Lo que vemos:** hay bastantes filas idénticas. Una fila repetida **no aporta información** y, peor, da un peso doble a ese paciente, sesgando cualquier estadística. Se eliminan sin dudar.


In [ ]:
filas_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Filas tras eliminar duplicados exactos:", len(df),
      " (eliminadas", filas_antes - len(df), ")")

# ¿Quedan pacientes repetidos por 'paciente_id' aunque la fila no sea idéntica?
print("paciente_id repetidos que aún quedan:", int(df["paciente_id"].duplicated().sum()))

**Lo que vemos:** tras quitar las filas idénticas, aún pueden quedar `paciente_id` repetidos con datos algo distintos (por ejemplo, una carga parcial y otra completa del mismo paciente). Estos son **más delicados**: no se borran a ciegas. Aplicamos una regla justificada: **quedarnos con el registro más completo** (el que tenga menos casillas vacías) de cada paciente.


In [ ]:
# Regla de resolución: por cada paciente, conservar la fila con MENOS valores ausentes
df["n_faltantes"] = df.isna().sum(axis=1)
df = (df.sort_values("n_faltantes")               # las más completas primero
        .drop_duplicates(subset="paciente_id", keep="first")
        .drop(columns="n_faltantes")
        .sort_values("paciente_id")
        .reset_index(drop=True))
print("Filas finales:", len(df))
print("¿paciente_id únicos?:", df["paciente_id"].is_unique)

**Lo que vemos:** cada paciente aparece ahora **una sola vez**, conservando su registro más informativo. El dataset está limpio. Hemos tomado, en cada paso, una **decisión clínica documentada**, no una receta mecánica.


## 10. El antes y el después: ¿ha merecido la pena?

Comparemos el punto de partida (`df_sucio`) con el resultado (`df`). Un buen resumen de limpieza muestra, lado a lado, las cifras que cambiaron: número de filas, categorías de las variables de texto y rangos de las numéricas clave.


In [ ]:
def rango(serie):
    s = pd.to_numeric(serie, errors="coerce")
    return f"{s.min():.1f} – {s.max():.1f}"

print("                         ANTES (sucio)                    DESPUÉS (limpio)")
print("Filas                   ", f"{len(df_sucio):<28}", len(df))
print("Categorías de 'sexo'    ", f"{len(df_sucio['sexo'].unique()):<28}", df["sexo"].nunique())
print("Categorías 'tabaquismo' ", f"{df_sucio['tabaquismo'].nunique():<28}", df["tabaquismo"].nunique())
print("Rango de 'glucemia'     ", f"{'texto (no calculable)':<28}", rango(df["glucemia"]))
print("Rango de 'edad'         ", f"{rango(df_sucio['edad']):<28}", rango(df["edad"]))
print("Rango de 'imc'          ", f"{rango(df_sucio['imc']):<28}", rango(df["imc"]))
print("Nulos totales           ", f"{int(df_sucio.isna().sum().sum()):<28}", int(df.isna().sum().sum()))

**Lo que vemos:** el contraste es rotundo. `sexo` pasó de media docena de categorías a **2**; `tabaquismo`, a **3**. La `glucemia` pasó de ser **texto ilegible** a un rango numérico coherente. Las edades e IMC imposibles desaparecieron. Y no queda **ningún** valor ausente. Los mismos datos, ahora **utilizables**.


### Los histogramas finales, ya sobre datos limpios

Cerramos el EDA "viendo" dos variables clave sobre el dataset limpio. Un histograma cuenta cuántos pacientes caen en cada tramo. Empezamos por la **edad**.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["edad"], bins=30, color="#00AEC7", edgecolor="white")
plt.xlabel("Edad (años)")
plt.ylabel("Nº de pacientes")
plt.title("Distribución de la edad (dataset limpio, datos sintéticos)")
plt.show()

**Lo que vemos:** una cohorte de adultos, con más pacientes de mediana edad y mayores que jóvenes, y **sin rastro** de aquellas edades imposibles. Es una pirámide poblacional creíble para un servicio clínico. Veamos ahora la **glucemia** corregida.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["glucemia"], bins=50, color="#00AEC7", edgecolor="white")
plt.axvline(126, color="#E4572E", linestyle="--", label="umbral diabetes (126 mg/dL)")
plt.xlabel("Glucemia (mg/dL)")
plt.ylabel("Nº de pacientes")
plt.title("Distribución de la glucemia (dataset limpio, datos sintéticos)")
plt.legend()
plt.show()

**Lo que vemos:** una única distribución centrada en la normalidad (~90–110 mg/dL), con una cola a la derecha que recoge a los pacientes con hiperglucemia y diabetes (por encima de 126 mg/dL). Sin la limpieza de unidades, esta gráfica habría sido un sinsentido con dos montañas separadas, y cualquier media o umbral calculado sobre ella, falso.


## 11. Por qué esto importa tanto en salud: el sesgo se hereda

Detengámonos en la idea que da sentido a todo el cuaderno.

> 💡 **La idea que hay que martillar.** Un modelo **aprende del histórico**. Si el histórico está **sucio o sesgado**, el modelo **hereda el sesgo**. No es un fallo técnico que arregle un algoritmo mejor: es un **espejo**.

Lo hemos tocado con las manos en el paso de los nulos. Al `hdl` le faltaban datos **sobre todo en los jóvenes**. Imaginemos que no lo hubiéramos mirado y hubiéramos rellenado con la mediana global, o borrado esas filas:

- Con la **mediana global**, todos los jóvenes habrían recibido un HDL "de adulto medio": el modelo los vería a todos parecidos y **peor caracterizados**.
- **Borrando** esas filas, los jóvenes casi **desaparecen** del dataset, y el modelo sería sistemáticamente peor con ellos —justo con quien menos datos había—.

En ambos casos, el resultado es un modelo que **funciona peor para un grupo concreto de pacientes**. Eso no es solo "calidad de dato": es un problema de **equidad clínica**. La misma lógica se aplica a las categorías mal codificadas (un gradiente de riesgo que se diluye) o a las unidades mezcladas (una analítica traducida al revés).

> ⚠️ En una historia clínica **real**, estos problemas aparecen a gran escala: laboratorios de distintos centros en **unidades diferentes**, categorías codificadas de mil formas, sensores con lecturas imposibles, y —lo más serio— **huecos que no son aleatorios** porque reflejan quién accede menos al sistema. **La calidad del dato es el techo de la calidad del modelo:** ningún algoritmo, por sofisticado que sea, compensa un histórico sucio o sesgado. Y en salud, ese sesgo se traduce en **desigualdad ante el paciente**.


## 12. Qué hemos aprendido

- **Mirar antes de tocar.** `shape`, `head`, `dtypes` y `describe` son el primer diagnóstico. Una columna numérica que aparece como **texto** ya delata un problema.
- **Normalizar el formato ≠ interpretar el significado.** Primero unificamos (`strip().lower()`), después mapeamos con una **regla de dominio explícita** (sexo → M/F, tabaquismo → nunca/ex/activo).
- **El criterio clínico es insustituible.** Solo quien conoce la fisiología ve que una glucemia de 5,5 está en **mmol/L** y hay que multiplicar por 18, o que una edad de 250 es imposible.
- **No todos los nulos son iguales.** Un hueco **no aleatorio** (HDL en jóvenes) exige imputación **con criterio** —informada por edad y sexo—, no la mediana global, y **debe documentarse**.
- **Limpiar es una decisión clínica con consecuencias de equidad.** Un histórico sucio o sesgado produce un modelo sesgado. La calidad del dato es el **techo** del modelo.

Con los datos ya entendidos y limpios, el siguiente paso —antes de entrenar ningún modelo "de verdad"— es aprender a **evaluarlo con honestidad**. Eso es la **U3**.


### El atajo de 2026: pídeselo al asistente (y léelo con criterio)

No tienes que recordar la sintaxis de nada de esto. En el curso, el **criterio clínico lo pones tú** y el **código lo escribe la IA**. Podrías describir la tarea a un asistente (el de Colab, Claude, u otro) con un *prompt* preciso como este:

```
Tengo 'pacientes_sucio.csv' (datos SINTÉTICOS). Límpialo en español, por
celdas y comentado, aplicando REGLAS DE DOMINIO CLÍNICO:
- 'sexo': normaliza (minúsculas, sin espacios) y mapea a {M, F}.
- 'tabaquismo': normaliza y mapea a {nunca, ex, activo}.
- 'glucemia': pásala a numérica aceptando coma decimal; detecta los valores
  en mmol/L (los bajos, <30) y conviértelos a mg/dL multiplicando por 18.
- Convierte a numérico las columnas con texto ('desconocido','N/D'); lo no
  válido -> nulo.
- 'hdl' ausente: NO uses la mediana global (falta más en jóvenes); imputa
  informado por edad y sexo, y DOCUMENTA el posible sesgo.
- Descarta valores imposibles: edad en [0,120], tensión > 0, imc en rango humano.
- Elimina duplicados exactos; para 'paciente_id' repetidos, quédate con el
  registro más completo. No pises el fichero original.
Muéstrame un antes/después (nº de filas, categorías y rangos).
```

**Tu papel** no es teclear el código, sino **saber qué pedir y revisar el resultado**: ¿resolvió bien el sexo ambiguo? ¿aplicó el ×18 al grupo correcto? ¿trató los nulos con conciencia del sesgo? Y aún hay un paso más: si el asistente puede *generar* esta limpieza, también puede **mejorarla iterando** a partir de los resultados —el ciclo **planificar → actuar → observar** que has usado sin nombrarlo—. Ese *bucle de auto-mejora* lo abordamos a fondo en la **U10**.
